# 🫁 Respiratory Sound Classification
## Stage 2 — Dataset Exploration & Visualization

> **Dataset:** ICBHI 2017 Respiratory Sound Database  
> **Path:** `C:\Users\User\Desktop\respiratory  -audio- datasets`  
> **Purpose:** Educational and research use only. NOT for medical diagnosis.

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Styling ───────────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='darkgrid', palette='muted')

print('✅ Libraries loaded.')

In [ ]:
# ── Dataset Paths ─────────────────────────────────────────────────────────────
DATASET_ROOT  = r'C:\Users\User\Desktop\respiratory  -audio- datasets'
AUDIO_DIR     = os.path.join(DATASET_ROOT,
                             'Respiratory_Sound_Database',
                             'Respiratory_Sound_Database',
                             'audio_and_txt_files')
DIAGNOSIS_CSV = os.path.join(DATASET_ROOT,
                             'Respiratory_Sound_Database',
                             'Respiratory_Sound_Database',
                             'patient_diagnosis.csv')
DEMOGRAPHIC_TXT = os.path.join(DATASET_ROOT, 'demographic_info.txt')

# Output directories
GRAPHS_DIR  = r'c:\Users\User\Desktop\NLP PROJECT\results\graphs'
REPORTS_DIR = r'c:\Users\User\Desktop\NLP PROJECT\results\reports'
os.makedirs(GRAPHS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Class label mapping
CLASS_MAP = {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}
CLASS_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
CLASS_COLORS = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

print(f'✅ Dataset root   : {DATASET_ROOT}')
print(f'✅ Audio directory: {AUDIO_DIR}')
print(f'✅ WAV files found : {len(glob.glob(os.path.join(AUDIO_DIR, "*.wav")))}')
print(f'✅ TXT files found : {len(glob.glob(os.path.join(AUDIO_DIR, "*.txt")))}')

---
## 1 · Parse All Annotations into a Master DataFrame

In [ ]:
def parse_annotation_file(txt_path):
    """Parse one ICBHI .txt annotation file into a list of cycle dicts."""
    records = []
    fname   = os.path.basename(txt_path).replace('.txt', '')
    parts   = fname.split('_')
    patient_id = int(parts[0])
    location   = parts[2]
    mode       = parts[3]
    equipment  = parts[4] if len(parts) > 4 else 'Unknown'

    with open(txt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cols = line.split('\t')
            if len(cols) < 4:
                continue
            start, end, crackle, wheeze = (
                float(cols[0]), float(cols[1]),
                int(cols[2]),   int(cols[3])
            )
            label_idx = CLASS_MAP[(crackle, wheeze)]
            records.append({
                'patient_id' : patient_id,
                'filename'   : fname,
                'location'   : location,
                'mode'       : mode,
                'equipment'  : equipment,
                'start'      : start,
                'end'        : end,
                'duration'   : round(end - start, 4),
                'crackle'    : crackle,
                'wheeze'     : wheeze,
                'label'      : label_idx,
                'label_name' : CLASS_NAMES[label_idx],
                'wav_path'   : os.path.join(AUDIO_DIR, fname + '.wav'),
            })
    return records


# Parse every annotation file
all_records = []
for txt_path in sorted(glob.glob(os.path.join(AUDIO_DIR, '*.txt'))):
    all_records.extend(parse_annotation_file(txt_path))

cycles_df = pd.DataFrame(all_records)

print(f'Total respiratory cycles parsed : {len(cycles_df):,}')
print(f'Unique patients                 : {cycles_df["patient_id"].nunique()}')
print(f'Unique recording files          : {cycles_df["filename"].nunique()}')
cycles_df.head()

In [ ]:
# ── Merge patient diagnosis info ─────────────────────────────────────────────
diag_df = pd.read_csv(DIAGNOSIS_CSV, header=None, names=['patient_id', 'diagnosis'])
cycles_df = cycles_df.merge(diag_df, on='patient_id', how='left')

print('Diagnosis columns merged.')
cycles_df[['patient_id', 'label_name', 'diagnosis', 'equipment']].head(10)

---
## 2 · Summary Statistics

In [ ]:
print('=' * 55)
print('  ICBHI 2017 — DATASET SUMMARY')
print('=' * 55)
print(f'  Total cycles        : {len(cycles_df):,}')
print(f'  Total patients      : {cycles_df["patient_id"].nunique()}')
print(f'  Total WAV files     : {cycles_df["filename"].nunique()}')
print(f'  Total audio (hrs)   : {cycles_df["duration"].sum() / 3600:.2f}')
print(f'  Avg cycle duration  : {cycles_df["duration"].mean():.2f}s')
print(f'  Min cycle duration  : {cycles_df["duration"].min():.2f}s')
print(f'  Max cycle duration  : {cycles_df["duration"].max():.2f}s')
print()
print('  CLASS DISTRIBUTION')
print('  ' + '-' * 45)
class_counts = cycles_df['label_name'].value_counts().reindex(CLASS_NAMES)
for name, count in class_counts.items():
    pct = count / len(cycles_df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {name:<8} : {count:>5}  ({pct:5.1f}%)  {bar}')
print()
print('  EQUIPMENT DISTRIBUTION')
print('  ' + '-' * 45)
for eq, cnt in cycles_df['equipment'].value_counts().items():
    print(f'  {eq:<12}: {cnt:>5} cycles')

---
## 3 · Class Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ICBHI 2017 — Class Distribution', fontsize=15, fontweight='bold')

# Bar chart
ax = axes[0]
bars = ax.bar(CLASS_NAMES, class_counts.values, color=CLASS_COLORS, edgecolor='white', linewidth=1.2)
ax.set_title('Respiratory Cycle Count per Class')
ax.set_ylabel('Number of Cycles')
ax.set_xlabel('Sound Class')
for bar, v in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{v:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    class_counts.values, labels=CLASS_NAMES,
    colors=CLASS_COLORS, autopct='%1.1f%%',
    startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontweight('bold')
    t.set_fontsize(11)
ax2.set_title('Class Proportion')

plt.tight_layout()
save_path = os.path.join(GRAPHS_DIR, 'class_distribution.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 4 · Cycle Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Respiratory Cycle Duration Analysis', fontsize=15, fontweight='bold')

# Overall histogram
ax = axes[0]
ax.hist(cycles_df['duration'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(cycles_df['duration'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {cycles_df["duration"].mean():.2f}s')
ax.set_title('All Cycles — Duration Histogram')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.legend()

# Per-class box plot
ax2 = axes[1]
data_per_class = [cycles_df[cycles_df['label_name'] == c]['duration'].values for c in CLASS_NAMES]
bp = ax2.boxplot(data_per_class, patch_artist=True, notch=False,
                 medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], CLASS_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax2.set_xticklabels(CLASS_NAMES)
ax2.set_title('Duration Distribution per Class')
ax2.set_xlabel('Sound Class')
ax2.set_ylabel('Duration (seconds)')

plt.tight_layout()
save_path = os.path.join(GRAPHS_DIR, 'cycle_duration_distribution.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 5 · Patient and Diagnosis Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Patient-Level Analysis', fontsize=15, fontweight='bold')

# Diagnosis breakdown
ax = axes[0]
diag_counts = cycles_df.drop_duplicates('patient_id')['diagnosis'].value_counts()
colors_diag = sns.color_palette('Set2', len(diag_counts))
bars = ax.barh(diag_counts.index, diag_counts.values, color=colors_diag)
ax.set_title('Patient Count by Diagnosis')
ax.set_xlabel('Number of Patients')
for bar, v in zip(bars, diag_counts.values):
    ax.text(v + 0.3, bar.get_y() + bar.get_height() / 2, str(v), va='center', fontweight='bold')

# Cycles per patient (sorted)
ax2 = axes[1]
cycles_per_patient = cycles_df.groupby('patient_id').size().sort_values(ascending=False)
ax2.bar(range(len(cycles_per_patient)), cycles_per_patient.values, color='steelblue', alpha=0.8)
ax2.axhline(cycles_per_patient.mean(), color='red', linestyle='--', linewidth=1.5,
            label=f'Mean = {cycles_per_patient.mean():.1f}')
ax2.set_title('Respiratory Cycles per Patient')
ax2.set_xlabel('Patient (sorted by cycle count)')
ax2.set_ylabel('Number of Cycles')
ax2.legend()

plt.tight_layout()
save_path = os.path.join(GRAPHS_DIR, 'patient_analysis.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 6 · Equipment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Recording Equipment Analysis', fontsize=15, fontweight='bold')

eq_counts = cycles_df['equipment'].value_counts()

ax = axes[0]
ax.pie(eq_counts.values, labels=eq_counts.index,
       autopct='%1.1f%%', startangle=140,
       colors=sns.color_palette('Set1', len(eq_counts)),
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Cycles per Equipment')

# Class distribution per equipment
ax2 = axes[1]
pivot = cycles_df.groupby(['equipment', 'label_name']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=CLASS_NAMES)
pivot.plot(kind='bar', ax=ax2, color=CLASS_COLORS, edgecolor='white')
ax2.set_title('Class Distribution per Equipment')
ax2.set_xlabel('Equipment')
ax2.set_ylabel('Cycle Count')
ax2.tick_params(axis='x', rotation=30)
ax2.legend(title='Class')

plt.tight_layout()
save_path = os.path.join(GRAPHS_DIR, 'equipment_distribution.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 7 · Audio Waveform Visualization (One Sample per Class)

In [ ]:
SR_TARGET = 22050   # uniform sampling rate

def load_cycle(row, sr=SR_TARGET):
    """Load a single respiratory cycle segment from a WAV file."""
    y, sr_orig = librosa.load(row['wav_path'], sr=None, mono=True)
    if sr_orig != sr:
        y = librosa.resample(y, orig_sr=sr_orig, target_sr=sr)
    # Trim to cycle boundaries
    start_sample = int(row['start'] * sr)
    end_sample   = int(row['end']   * sr)
    y_cycle      = y[start_sample:end_sample]
    # Normalize amplitude
    if y_cycle.max() != 0:
        y_cycle = y_cycle / np.abs(y_cycle).max()
    return y_cycle


# Pick one representative sample per class
samples = {}
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    subset = cycles_df[
        (cycles_df['label'] == cls_idx) &
        (cycles_df['duration'] > 1.5) &
        (cycles_df['duration'] < 5.0)
    ]
    row = subset.sample(1, random_state=SEED).iloc[0]
    samples[cls_name] = (load_cycle(row), row)
    print(f'  {cls_name:<8}: {row["filename"]} | {row["duration"]:.2f}s | patient {row["patient_id"]}')

print('\n✅ Sample cycles loaded.')

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
fig.suptitle('Respiratory Cycle Waveforms — One Sample per Class', fontsize=14, fontweight='bold')

for ax, (cls_name, (y, row)) in zip(axes, samples.items()):
    times = np.linspace(0, len(y) / SR_TARGET, len(y))
    color = CLASS_COLORS[CLASS_NAMES.index(cls_name)]
    ax.plot(times, y, color=color, linewidth=0.6, alpha=0.9)
    ax.set_title(f'{cls_name}  —  Patient {row["patient_id"]}  |  {row["equipment"]}  |  {row["duration"]:.2f}s',
                 fontsize=11)
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, len(y) / SR_TARGET)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout(rect=[0, 0, 1, 0.97])
save_path = os.path.join(GRAPHS_DIR, 'waveforms_per_class.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 8 · Mel Spectrogram Visualization (One per Class)

In [ ]:
N_FFT   = 1024
HOP     = 256
N_MELS  = 64

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Mel Spectrograms — One Sample per Class', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, (cls_name, (y, row)) in zip(axes, samples.items()):
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=SR_TARGET, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)
    img = librosa.display.specshow(
        mel_db, sr=SR_TARGET, hop_length=HOP,
        x_axis='time', y_axis='mel', ax=ax,
        cmap='magma')
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set_title(f'{cls_name}  —  Patient {row["patient_id"]}', fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Mel Frequency')

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_path = os.path.join(GRAPHS_DIR, 'mel_spectrograms_per_class.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 9 · MFCC Visualization (One per Class)

In [ ]:
N_MFCC = 40

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('MFCC Features — One Sample per Class', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, (cls_name, (y, row)) in zip(axes, samples.items()):
    mfcc = librosa.feature.mfcc(y=y, sr=SR_TARGET, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP)
    img = librosa.display.specshow(
        mfcc, sr=SR_TARGET, hop_length=HOP,
        x_axis='time', ax=ax, cmap='coolwarm')
    fig.colorbar(img, ax=ax)
    ax.set_title(f'{cls_name}  —  Patient {row["patient_id"]}', fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('MFCC Coefficient')

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_path = os.path.join(GRAPHS_DIR, 'mfcc_per_class.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 10 · Spectral Feature Comparison

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Spectral Features — All Classes Overlaid', fontsize=14, fontweight='bold')

for cls_name, (y, row) in samples.items():
    color = CLASS_COLORS[CLASS_NAMES.index(cls_name)]
    times = librosa.frames_to_time(
        np.arange(librosa.feature.spectral_centroid(y=y, sr=SR_TARGET, n_fft=N_FFT, hop_length=HOP).shape[1]),
        sr=SR_TARGET, hop_length=HOP)

    # Spectral Centroid
    cent = librosa.feature.spectral_centroid(y=y, sr=SR_TARGET, n_fft=N_FFT, hop_length=HOP)[0]
    t = librosa.frames_to_time(np.arange(len(cent)), sr=SR_TARGET, hop_length=HOP)
    axes[0].plot(t, cent, label=cls_name, color=color, linewidth=1.5, alpha=0.85)

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0]
    t2  = librosa.frames_to_time(np.arange(len(zcr)), sr=SR_TARGET, hop_length=HOP)
    axes[1].plot(t2, zcr, label=cls_name, color=color, linewidth=1.5, alpha=0.85)

    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=SR_TARGET, n_fft=N_FFT, hop_length=HOP)[0]
    t3      = librosa.frames_to_time(np.arange(len(rolloff)), sr=SR_TARGET, hop_length=HOP)
    axes[2].plot(t3, rolloff, label=cls_name, color=color, linewidth=1.5, alpha=0.85)

axes[0].set_title('Spectral Centroid'); axes[0].set_ylabel('Hz'); axes[0].legend()
axes[1].set_title('Zero Crossing Rate');  axes[1].set_ylabel('Rate'); axes[1].legend()
axes[2].set_title('Spectral Rolloff');    axes[2].set_ylabel('Hz');  axes[2].legend()
axes[2].set_xlabel('Time (seconds)')

for ax in axes:
    ax.set_xlim(left=0)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_path = os.path.join(GRAPHS_DIR, 'spectral_features_comparison.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 11 · Patient-Level Split (Train / Val / Test — No Data Leakage)

In [ ]:
from sklearn.model_selection import train_test_split

# Get unique patients
all_patients = cycles_df['patient_id'].unique()
np.random.seed(SEED)
np.random.shuffle(all_patients)

# 80 / 10 / 10 split at patient level
train_patients, temp_patients = train_test_split(all_patients, test_size=0.2, random_state=SEED)
val_patients,   test_patients  = train_test_split(temp_patients, test_size=0.5, random_state=SEED)

train_df = cycles_df[cycles_df['patient_id'].isin(train_patients)].copy()
val_df   = cycles_df[cycles_df['patient_id'].isin(val_patients)].copy()
test_df  = cycles_df[cycles_df['patient_id'].isin(test_patients)].copy()

print('Patient-Level Split:')
print(f'  Train : {len(train_patients)} patients | {len(train_df):,} cycles')
print(f'  Val   : {len(val_patients)}  patients | {len(val_df):,} cycles')
print(f'  Test  : {len(test_patients)}  patients | {len(test_df):,} cycles')
print()

# Verify no overlap
assert len(set(train_patients) & set(val_patients)) == 0, 'Leakage between train and val!'
assert len(set(train_patients) & set(test_patients)) == 0, 'Leakage between train and test!'
assert len(set(val_patients)   & set(test_patients)) == 0, 'Leakage between val and test!'
print('✅ No patient overlap — zero data leakage confirmed.')

# Class distribution per split
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    counts = df['label_name'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
    print(f'\n  [{name}] ' + ' | '.join([f'{c}: {v}' for c, v in counts.items()]))

In [ ]:
# Visualize split distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Class Distribution per Split (Patient-Level)', fontsize=14, fontweight='bold')

for ax, (split_name, split_df) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
    counts = split_df['label_name'].value_counts().reindex(CLASS_NAMES).fillna(0)
    bars = ax.bar(CLASS_NAMES, counts.values, color=CLASS_COLORS, edgecolor='white')
    ax.set_title(split_name)
    ax.set_xlabel('Class')
    ax.set_ylabel('Cycles')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(int(v)), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
save_path = os.path.join(GRAPHS_DIR, 'split_distribution.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

---
## 12 · Save Processed Metadata for Stage 3

In [ ]:
# Save master DataFrame
cycles_df.to_csv(os.path.join(REPORTS_DIR, 'all_cycles.csv'), index=False)

# Save splits
train_df.to_csv(os.path.join(REPORTS_DIR, 'train_cycles.csv'), index=False)
val_df.to_csv(os.path.join(REPORTS_DIR,   'val_cycles.csv'),   index=False)
test_df.to_csv(os.path.join(REPORTS_DIR,  'test_cycles.csv'),  index=False)

print('✅ Saved:')
print(f'  {REPORTS_DIR}\\all_cycles.csv')
print(f'  {REPORTS_DIR}\\train_cycles.csv')
print(f'  {REPORTS_DIR}\\val_cycles.csv')
print(f'  {REPORTS_DIR}\\test_cycles.csv')
print()
print('Stage 2 Complete. Ready for Stage 3: Preprocessing & Feature Extraction.')